In [1]:
from google import genai
from IPython.display import display, Markdown
from dotenv import load_dotenv

load_dotenv()
client =genai.Client()

interaction = client.interactions.create(
    model= "gemini-3.7-flash",
    input="請逐步教學含視窗化操作指令, 如何使用raspberry樹莓派架設出家用的NAS伺服器?")
display(Markdown(interaction.output_text))
      

使用樹莓派（Raspberry Pi）架設家用 NAS 最推薦的方案是使用 **OpenMediaVault (簡稱 OMV)**。它是一套專為 NAS 設計的開源系統，安裝完成後**所有操作都可以在瀏覽器（視窗化介面）完成**，支援 Windows (Samba)、Mac 和手機存取。

以下為您整理「從零開始」的完整圖文級步驟教學：

---

### 步驟一：準備硬體與軟體

#### 必備硬體：
1. **樹莓派**：建議 Raspberry Pi 4B 或 5（具備 USB 3.0 與 Gigabit 網路孔，效能最佳）。
2. **MicroSD 卡**：16GB 以上（建議 Class 10/A1 以上）。
3. **外接硬碟**：存放資料用（建議使用「有獨立供電」的 USB 3.0 外接硬碟或外接盒）。
4. **網路線**：建議插實體網路線以維持傳輸穩定。
5. **電腦一台**：用於燒錄系統與後續設定。

#### 必備軟體：
* 下載並安裝 [Raspberry Pi Imager](https://www.raspberrypi.com/software/)（官方燒錄工具）。

---

### 步驟二：燒錄樹莓派系統 (Lite 輕量版)

OMV 建議安裝在沒有桌面環境的 Lite 系統上，以維持最高效能。

1. 將 MicroSD 卡插入電腦，開啟 **Raspberry Pi Imager**。
2. **選擇裝置 (Device)**：選擇你的樹莓派型號（如 Raspberry Pi 4）。
3. **選擇作業系統 (Operating System)**：
   * 點選 `Raspberry Pi OS (other)` -> 選擇 `Raspberry Pi OS Lite (64-bit)`。
4. **選擇儲存卡 (Storage)**：選擇你的 MicroSD 卡。
5. 點擊「下一步」，跳出客製化設定視窗時點選 **「編輯設定」**：
   * **一般**：設定主機名稱（如 `nas`）、使用者名稱與密碼（例如 `pi` / `你的密碼`）。
   * **服務**：勾選 **「啟用 SSH」**，並選擇「使用密碼驗證」。
6. 儲存設定後，點擊「寫入」並等待燒錄完成。

---

### 步驟三：安裝 OpenMediaVault (OMV) 核心

1. 將 SD 卡插回樹莓派，接上網路線並通電開機。
2. 查詢樹莓派的 IP 位址（可透過家用路由器後台查詢，例如 `192.168.1.100`）。
3. 在你的電腦上打開終端機（Windows 請開啟 `PowerShell` 或 `CMD`，Mac 打開 `Terminal`）。
4. 輸入以下指令連線至樹莓派（請將 `pi` 和 `IP` 替換為你的設定）：
   ```bash
   ssh pi@192.168.1.100
   ```
5. 輸入密碼登入後，輸入以下指令進行系統更新並一鍵安裝 OMV（安裝需時 15~30 分鐘，請耐心等候）：
   ```bash
   wget -O - https://github.com/OpenMediaVault-Plugin-Developers/installScript/raw/master/install | sudo bash
   ```
6. 安裝完成後，樹莓派會自動重新開機。**指令操作至此結束，接下來全為視窗化操作！**

---

### 步驟四：登入 OMV 視窗化後台

1. 在電腦瀏覽器輸入樹莓派的 IP（例如：`http://192.168.1.100`）。
2. 你會看到 OMV 的登入視窗：
   * **預設帳號**：`admin`
   * **預設密碼**：`openmediavault`
3. 登入後，首要任務是**更改預設密碼**：
   * 點擊右上角「人頭圖示」 -> 選擇「變更密碼」-> 修改並儲存。

> ⚠️ **重要提示**：OMV 每次更改設定，畫面上方都會出現**黃色橫幅**詢問「套用變更？」，**請務必點擊右上角的「✔ (套用)」**，設定才會真正生效。

---

### 步驟五：視窗化設定——掛載外接硬碟

將外接硬碟插入樹莓派的 **藍色 USB 3.0 孔**。

1. **檢查硬碟**：
   * 到左側選單：`儲存區 (Storage)` -> `磁碟 (Disks)`，確認能看到你的外接硬碟。
2. **建立檔案系統 (格式化)**：
   * 到 `儲存區 (Storage)` -> `檔案系統 (File Systems)`。
   * 點擊左上角 **「＋」圖示** -> 選擇 **「掛載 (Mount)」**（若硬碟內已有資料）。
   * *若是新硬碟需要格式化*：點擊 **「＋」** -> 選擇 **「建立並掛載 (Create and mount)」** -> 選擇外接硬碟 -> 檔案系統建議選擇 `EXT4` 或 `BTRFS` -> 點擊儲存。
   * 記得點擊上方黃色橫幅的 **「✔ (套用)」**。

---

### 步驟六：視窗化設定——建立使用者與共享資料夾

#### 1. 建立使用者帳號
1. 左側選單：`使用者 (Users)` -> `使用者 (Users)`。
2. 點擊 **「＋ (建立)」**：
   * **名稱**：例如 `user1`
   * **密碼**：設定一組密碼（未來在電腦連線時使用）。
3. 點擊「儲存」並套用變更。

#### 2. 建立共享資料夾 (Shared Folders)
1. 左側選單：`儲存區 (Storage)` -> `共用資料夾 (Shared Folders)`。
2. 點擊 **「＋ (建立)」**：
   * **名稱**：例如 `Public` 或 `Data`。
   * **檔案系統**：選擇剛才掛載的外接硬碟。
   * **權限 (Permissions)**：通常選擇 `系統管理員讀寫，使用者讀寫，其他唯讀` (或預設即可)。
3. 點擊「儲存」並套用變更。

#### 3. 開啟 SMB 網路共享 (讓 Windows/Mac 可以存取)
1. 左側選單：`服務 (Services)` -> `SMB/CIFS` -> `設定 (Settings)`。
   * 勾選 **「啟用 (Enabled)」** -> 點擊「儲存」。
2. 切換到上方分頁的 **「共享 (Shares)」**：
   * 點擊 **「＋ (建立)」**。
   * **共用資料夾**：選擇剛才建立的 `Public`。
   * 勾選 **「公開 (Public)」**（若希望免密碼存取）或保持預設（需要輸入帳密）。
   * 點擊「儲存」並套用變更。

---

### 步驟七：在電腦端連線你的 NAS

#### Windows 電腦連線：
1. 打開「檔案總管」，在上方網址列輸入：`\\192.168.1.100`（替換為樹莓派 IP）。
2. 按 Enter 後，會彈出視窗要求輸入帳號密碼：
   * 輸入步驟六建立的帳號密碼（如 `user1`）。
3. 就可以看到共享資料夾，能像一般隨身碟一樣拖拉複製檔案了！
4. *(進階技巧)*：在資料夾上按右鍵 -> 選擇「連線網路磁碟機」，可將 NAS 固定為電腦裡的 `Z:` 槽。

#### Mac 電腦連線：
1. 打開 Finder，上方選單點選 `前往` -> `連接伺服器...`（快捷鍵 `Cmd + K`）。
2. 輸入：`smb://192.168.1.100`。
3. 選擇「註冊使用者」，輸入帳號密碼即可連線掛載。

#### 手機存取 (iOS / Android)：
* **iOS**：內建「檔案」App -> 右上角 `...` -> `連接伺服器` -> 輸入 `smb://192.168.1.100`。
* **Android**：下載支援 SMB 的檔案總管（如 `CX 檔案總管` 或 `Solid Explorer`）-> 新增區域網路/SMB 裝置即可。